# INT4096 · Learning Machine Learning Through Financial Forecasting
## Student Colab Workbook — 60-Minute Guided Lesson

> **Course:** INT4096 — Artificial Intelligence and Machine Learning in Education  
> **Institution:** The Education University of Hong Kong  
> **Target audience:** Senior secondary / first-year university students new to machine learning

---

### How to use this notebook

| Symbol | Meaning |
|--------|----------|
| 🎯 **Task** | A question or coding challenge for you to complete |
| 💡 **Hint** | A prompt to get you unstuck |
| 📖 **Concept** | Background explanation |
| ✅ **Teacher Note** | Hidden reference answer (collapsed by default) |

**Run cells from top to bottom.** Each section builds on the one before it.

---

⚠️ **Important:** This notebook works with the **validation set only**. The final 20% (test block) is sealed by Decision D7 and is not unblinded here. All metrics you compute will be validation-set metrics.


---
## 🔧 Setup — Run this first
Install dependencies and load the frozen dataset snapshots.

In [ ]:
# Install required packages
!pip install scikit-learn pandas numpy matplotlib --quiet

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score

plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

RANDOM_STATE = 42
print('✅ Imports complete.')

In [ ]:
# ── Data loader ──────────────────────────────────────────────────────────────
# In the full project, data is fetched from:
#   AAPL: StockAnalysis API  GET https://stockanalysis.com/api/symbol/s/aapl/history
#   BTC:  Binance REST API   GET https://api.binance.com/api/v3/klines?symbol=BTCUSDT&interval=1d
# Here we simulate a small frozen slice for the lesson.

np.random.seed(RANDOM_STATE)

def _simulate_prices(n, start_price, daily_vol):
    """Simulate a geometric random walk (log-normal daily returns)."""
    log_returns = np.random.normal(0.0003, daily_vol, n)
    prices = start_price * np.exp(np.cumsum(log_returns))
    highs  = prices * (1 + np.abs(np.random.normal(0, 0.005, n)))
    lows   = prices * (1 - np.abs(np.random.normal(0, 0.005, n)))
    volume = np.random.lognormal(18, 0.5, n)  # shares/units
    return pd.DataFrame({'close': prices, 'high': highs, 'low': lows, 'volume': volume})

N_TOTAL = 500   # simplified lesson dataset
TRAIN_END   = int(N_TOTAL * 0.60)
VAL_END     = int(N_TOTAL * 0.80)
# Note: boundary rows are dropped (see Act II in the story document)
TRAIN_SLICE = slice(0, TRAIN_END - 1)          # -1 boundary purge
VAL_SLICE   = slice(TRAIN_END, VAL_END - 1)    # -1 boundary purge
# TEST_SLICE  = slice(VAL_END, N_TOTAL)         # 🔒 SEALED — decision D7

raw_aapl = _simulate_prices(N_TOTAL + 1, start_price=175.0, daily_vol=0.013)
raw_btc  = _simulate_prices(N_TOTAL + 1, start_price=42000.0, daily_vol=0.028)

print(f'Dataset size  : {N_TOTAL} rows')
print(f'Train rows    : {TRAIN_END - 1}  (60% minus boundary row)')
print(f'Validate rows : {VAL_END - TRAIN_END - 1}  (20% minus boundary row)')
print(f'Test rows     : 🔒 SEALED (20%) — not used in this notebook')
print('\nAAPL price range : ${:.2f} – ${:.2f}'.format(raw_aapl.close.min(), raw_aapl.close.max()))
print('BTC  price range : ${:,.0f} – ${:,.0f}'.format(raw_btc.close.min(), raw_btc.close.max()))

---
## 📖 Concept: The High-Score Trap

Consider this claim:

> *"Our model scored R² = **0.9556** on Apple validation data!"*

Impressive, right? But what if copying yesterday's close price into tomorrow's forecast *also* scores **R² = 0.9555**?

That's exactly what happened in our experiment. The gap between a sophisticated ML model and a "do nothing" rule was **0.0001**.

Why does this happen? Consecutive stock prices are highly correlated in *level*, so any forecast that stays near the previous price tracks the variation well — even when it misses every single daily *change*.

**This notebook teaches you to ask three questions about any AI score:**
1. What information could the model see at prediction time?
2. How well does a simple rule do on the same examples?
3. How certain is the improvement?


---
## 🎯 Task 1 · Build the Naive Persistence Baseline
*Estimated time: 10 minutes*

**The simplest possible forecast:** predict that tomorrow's close equals today's close.

$$\hat{P}_{t+1} = P_t \quad (\text{"just copy the latest price"})$$

Before you code anything, write your prediction below:

> **❓ Your prediction:** On the AAPL validation set, what do you think the R² of naive persistence will be — close to 0, 0.5, or close to 1? Why?

*Write your answer here before running the cells below.*

In [ ]:
# ── Task 1: Implement naive persistence ──────────────────────────────────────
# df has columns: close, high, low, volume
# The forecast for row t+1 is the CLOSE of row t.
#
# Your job:
#   1. Extract the validation close prices (use VAL_SLICE on raw_aapl)
#   2. Build the persistence forecast (shift by 1)
#   3. Compute MAE and R²
#   4. Plot actual vs forecast

# --- YOUR CODE HERE ---
val_close   = raw_aapl['close'].iloc[TRAIN_END: VAL_END].values
# Persistence: use the previous row's close as forecast
# HINT: val_close[:-1] are the "known" closes, val_close[1:] are the actuals

persistence_preds = None   # ← replace with your implementation
actuals           = None   # ← the corresponding actual closes

# mae_persistence = mean_absolute_error(actuals, persistence_preds)
# r2_persistence  = r2_score(actuals, persistence_preds)
# print(f'Persistence MAE : ${mae_persistence:.4f}')
# print(f'Persistence R²  : {r2_persistence:.4f}')
# --- END YOUR CODE ---

In [ ]:
# ✅ TEACHER REFERENCE ANSWER (hidden by default — collapse this cell)
val_close         = raw_aapl['close'].iloc[TRAIN_END: VAL_END].values
persistence_preds = val_close[:-1]   # today's close → tomorrow's forecast
actuals           = val_close[1:]    # tomorrow's actual close

mae_persistence = mean_absolute_error(actuals, persistence_preds)
r2_persistence  = r2_score(actuals, persistence_preds)
print(f'Persistence MAE : ${mae_persistence:.4f}')
print(f'Persistence R²  : {r2_persistence:.4f}')

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
t = np.arange(len(actuals))
axes[0].plot(t, actuals, label='Actual', lw=1.2)
axes[0].plot(t, persistence_preds, label='Persistence', lw=1.0, ls='--', alpha=0.7)
axes[0].set_title('AAPL: Actual vs Persistence Forecast')
axes[0].set_xlabel('Validation day')
axes[0].set_ylabel('Price ($)')
axes[0].legend()

errors = actuals - persistence_preds
axes[1].hist(errors, bins=30, edgecolor='white')
axes[1].axvline(0, color='red', ls='--')
axes[1].set_title('Persistence Forecast Error Distribution')
axes[1].set_xlabel('Error ($)')
axes[1].set_ylabel('Count')
plt.tight_layout()
plt.show()

print('\n💡 Key observation:')
print(f'   R² is high ({r2_persistence:.4f}) because consecutive prices are strongly correlated in LEVEL.')
print('   But the forecast misses every daily CHANGE — the errors are symmetric around zero.')
print('   This is the "high-score trap": a high R² does not mean the model learned useful patterns.')

---
## 🎯 Task 2 · Spot the Illegal Feature (Data Leakage)
*Estimated time: 10 minutes*

📖 **Data leakage** occurs when a model sees information at training time that would not have been available at prediction time in the real world.

**The rule:** At prediction time (end of day $t$), you only know:
- Today's OHLCV (open, high, low, close, volume)
- All past observations

You do **NOT** know:
- Tomorrow's close, high, low, open, or volume
- Any statistic computed over the whole series (past + future)

Below are six proposed features. **Which ones are illegal?**

In [ ]:
# ── Task 2: Identify leakage ──────────────────────────────────────────────────
# Study the six feature definitions below.
# For each one, decide: LEGAL or ILLEGAL?
# Write your answers as a dict: {feature_id: 'LEGAL'} or {feature_id: 'ILLEGAL'}

close = raw_aapl['close'].values
vol   = raw_aapl['volume'].values

# Feature A: 1-period return (today vs yesterday)
feat_A = np.diff(close, prepend=np.nan) / close  # (close_t - close_{t-1}) / close_t

# Feature B: 5-period moving average gap, window ends at tomorrow
feat_B_raw = pd.Series(close).shift(-1).rolling(5).mean()  # centred forward
feat_B = (close - feat_B_raw.values) / close

# Feature C: 10-period volatility, window ends at today
ret = pd.Series(close).pct_change()
feat_C = ret.rolling(10).std().values

# Feature D: whole-series mean-normalised close
feat_D = (close - close.mean()) / close.std()   # uses ALL dates including future

# Feature E: (high - low) / close, all from today
feat_E = (raw_aapl['high'].values - raw_aapl['low'].values) / close

# Feature F: tomorrow's close shifted back by one ("look-ahead by 1 step")
feat_F = pd.Series(close).shift(-1).values

# --- YOUR ANSWERS ---
my_answers = {
    'A': '???',   # LEGAL or ILLEGAL?
    'B': '???',
    'C': '???',
    'D': '???',
    'E': '???',
    'F': '???',
}
print('Your answers:', my_answers)

In [ ]:
# ✅ TEACHER REFERENCE ANSWER
correct_answers = {
    'A': 'LEGAL',    # Uses only close_t and close_{t-1} — both available at time t
    'B': 'ILLEGAL',  # shift(-1) uses close_{t+1}, which is tomorrow — not available!
    'C': 'LEGAL',    # Rolling std of past 10 returns, window ends at today
    'D': 'ILLEGAL',  # close.mean() / .std() uses the ENTIRE series including future dates
    'E': 'LEGAL',    # High, low, close are all from today
    'F': 'ILLEGAL',  # Explicitly uses tomorrow's close — the most obvious leakage
}

print('Feature | Verdict  | Reason')
print('--------|----------|' + '-'*55)
reasons = {
    'A': 'Uses close[t] and close[t-1] — both known at time t',
    'B': 'shift(-1) moves future values into the present row',
    'C': 'Rolling window ends at today; only past 10 days used',
    'D': 'close.mean() / .std() uses whole series (train+future)',
    'E': 'OHLCV all from today — session-end data is available',
    'F': 'Directly passes tomorrow\'s close as a feature',
}
for feat, verdict in correct_answers.items():
    print(f'   {feat}    | {verdict:<8} | {reasons[feat]}')

print('\n⚠️  Illegal features B, D, F correspond to leakage types injected in the')
print('   project mutation tests: forward shift, whole-series statistic, and direct look-ahead.')
print('   The test harness caught all 6 mutation variants — including obfuscated versions.')

print('\n💡 Practical check: append absurd future prices to the raw series, then rebuild')
print('   features. Any legal feature should produce IDENTICAL values for all past dates.')

---
## 🎯 Task 3 · Build the 10 Legal Features and Train a Model
*Estimated time: 10 minutes*

📖 **The prediction target is a price change, not a price.**

$$\hat{P}_{t+1} = \underbrace{P_t}_{\text{known anchor}} + \underbrace{\widehat{\Delta P}_{t+1}}_{\text{model learns this}}$$

By predicting the *change* on top of the known price, we give every model the same anchor and ask it to add value beyond copying the last price.

**Your task:** Complete the `build_features()` function below, then train an SVR and compare it with persistence.

In [ ]:
# ── Task 3: Feature engineering ───────────────────────────────────────────────

def build_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Build the 10 legal features from OHLCV data.
    All windows end at the CURRENT row (no look-ahead).

    Feature map (fill in the missing ones):
    ┌────────────────────────────┬────────────────────────────────────────────┬───────┐
    │ Group                      │ Construction                               │ Count │
    ├────────────────────────────┼────────────────────────────────────────────┼───────┤
    │ Returns                    │ % change in close over 1, 5, 20 periods    │ 3     │
    │ Moving-average gaps        │ % gap from 5- and 20-period avg close      │ 2     │
    │ Volatility                 │ 10- and 30-period std of 1-period returns  │ 2     │
    │ Price range                │ (high - low) / close                       │ 1     │
    │ Relative volume            │ % gap from 5-period avg volume             │ 1     │
    │ Calendar                   │ Day of week (0=Mon) — use index position   │ 1     │
    └────────────────────────────┴────────────────────────────────────────────┴───────┘
    """
    close = df['close']
    high  = df['high']
    low   = df['low']
    vol   = df['volume']
    ret1  = close.pct_change(1)

    features = pd.DataFrame(index=df.index)

    # ── Returns (3 features) ───────────────────────────────────────
    features['ret_1']  = close.pct_change(1)
    features['ret_5']  = close.pct_change(5)   # ← DONE
    features['ret_20'] = None  # ← YOUR CODE: % change over 20 periods

    # ── Moving-average gaps (2 features) ──────────────────────────
    ma5  = close.rolling(5).mean()
    ma20 = close.rolling(20).mean()
    features['ma_gap_5']  = (close - ma5)  / close
    features['ma_gap_20'] = None  # ← YOUR CODE: % gap from 20-period avg

    # ── Volatility (2 features) ────────────────────────────────────
    features['vol_10'] = ret1.rolling(10).std()
    features['vol_30'] = None  # ← YOUR CODE: 30-period std of 1-period returns

    # ── Price range (1 feature) ────────────────────────────────────
    features['range'] = (high - low) / close

    # ── Relative volume (1 feature) ───────────────────────────────
    vol_ma5 = vol.rolling(5).mean()
    features['rel_vol'] = None  # ← YOUR CODE: % gap from 5-period avg volume

    # ── Calendar (1 feature) ──────────────────────────────────────
    # Use the integer position modulo 7 as a day-of-week proxy
    features['day_of_week'] = np.arange(len(df)) % 7   # simplified (Mon=0)

    return features


# Build target: next-period price CHANGE (what models learn)
def build_target(df: pd.DataFrame) -> pd.Series:
    return df['close'].shift(-1) - df['close']  # ΔP_{t+1} = close_{t+1} - close_t


print('build_features and build_target defined.')
print('Complete the missing lines (None values), then run Task 3b below.')

In [ ]:
# ── Task 3b: Train SVR and compare with persistence ─────────────────────────

def prepare_splits(raw_df, train_end, val_end):
    """Build train/val feature matrices and target vectors."""
    X_all = build_features(raw_df)
    y_all = build_target(raw_df)

    # Drop last row (target is NaN after shift(-1))
    X_all = X_all.iloc[:-1]
    y_all = y_all.iloc[:-1]

    # Boundary purge: drop 1 row at each split boundary
    X_train = X_all.iloc[:train_end - 1].dropna()
    y_train = y_all.iloc[:train_end - 1].loc[X_train.index]

    X_val   = X_all.iloc[train_end: val_end - 1].dropna()
    y_val   = y_all.iloc[train_end: val_end - 1].loc[X_val.index]

    return X_train, y_train, X_val, y_val


def compute_skill(model_mae, persistence_mae):
    """Skill score: fraction of persistence MAE saved. Positive = better than naive."""
    return (1 - model_mae / persistence_mae) * 100  # in percent


# Prepare data
X_train, y_train, X_val, y_val = prepare_splits(raw_aapl, TRAIN_END, VAL_END)

# ── P2: Standardise using training data only ─────────────────────────────────
scaler  = StandardScaler().fit(X_train)
X_train_p2 = scaler.transform(X_train)
X_val_p2   = scaler.transform(X_val)

# Persistence baseline on validation close prices
val_close_arr = raw_aapl['close'].iloc[TRAIN_END: VAL_END].values
mae_naive     = mean_absolute_error(val_close_arr[1:], val_close_arr[:-1])
print(f'Persistence MAE  : ${mae_naive:.4f}')

# ── YOUR CODE: reconstruct forecast price from predicted change ──────────────
# The anchor is raw_aapl['close'] at the same validation rows
anchor_val = raw_aapl['close'].iloc[TRAIN_END: VAL_END - 1].values
actual_val = raw_aapl['close'].iloc[TRAIN_END + 1: VAL_END].values

# Train SVR (P2)
svr = SVR(kernel='rbf', C=10.0, epsilon=0.01)
svr.fit(X_train_p2, y_train)
delta_pred      = svr.predict(X_val_p2)
price_pred_svr  = anchor_val[:len(delta_pred)] + delta_pred
actual_trimmed  = actual_val[:len(delta_pred)]

mae_svr   = mean_absolute_error(actual_trimmed, price_pred_svr)
skill_svr = compute_skill(mae_svr, mae_naive)
print(f'SVR (P2) MAE     : ${mae_svr:.4f}')
print(f'SVR (P2) Skill   : {skill_svr:+.2f}%')
print()
print('💡 Positive skill = lower average error than persistence.')
print('   How does your SVR skill compare with the project result (+0.52% on AAPL)?')

In [ ]:
# ✅ TEACHER REFERENCE: Completed build_features (the missing lines)
# 
# features['ret_20'] = close.pct_change(20)
# features['ma_gap_20'] = (close - ma20) / close
# features['vol_30'] = ret1.rolling(30).std()
# features['rel_vol'] = (vol - vol_ma5) / vol_ma5
#
# Key teaching points:
# 1. All windows end at today (iloc[:t] or rolling with min_periods).
# 2. Relative features (pct changes, gaps) allow the SAME 10 features to work
#    for both AAPL (~$200) and BTC (~$60,000) without rescaling.
# 3. The target is ΔP (change), not P (level), so the model is not asked to
#    "know" the absolute price — it already has that anchor.
print('Completed feature reference printed above in comments.')

---
## 🎯 Task 4 · Compare All Models and Read the Bootstrap Intervals
*Estimated time: 10 minutes*

📖 **A skill score tells you whether a model beat persistence. But how certain is that gain?**

We use a **moving-block bootstrap** to quantify uncertainty:
- Pair model errors and persistence errors by date
- Resample blocks of 10 consecutive dates (keeps temporal autocorrelation)
- Compute skill in each of 1,000 resamples → 95% interval

**In the full project: all 20 bootstrap intervals included zero** — including the best run (+0.52%).

**Your task:** Run all 5 models, compute their skill scores, then run the bootstrap and plot the intervals.

In [ ]:
# ── Task 4a: Run all 5 models ─────────────────────────────────────────────────

models = {
    'Linear':          LinearRegression(),
    'Ridge':           Ridge(alpha=1.0, random_state=RANDOM_STATE),
    'SVR':             SVR(kernel='rbf', C=10.0, epsilon=0.01),
    'Random Forest':   RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE),
    'Hist. GBT':       HistGradientBoostingRegressor(random_state=RANDOM_STATE),
}

results = {}
for name, model in models.items():
    model.fit(X_train_p2, y_train)
    delta_hat    = model.predict(X_val_p2)
    n            = min(len(delta_hat), len(anchor_val), len(actual_val))
    price_hat    = anchor_val[:n] + delta_hat[:n]
    actual_n     = actual_val[:n]
    mae          = mean_absolute_error(actual_n, price_hat)
    skill        = compute_skill(mae, mae_naive)
    # Store paired errors for bootstrap
    model_errs   = np.abs(actual_n - price_hat)
    naive_errs   = np.abs(actual_n - anchor_val[:n])  # persistence errors on same dates
    results[name] = {'mae': mae, 'skill': skill, 'model_errs': model_errs, 'naive_errs': naive_errs}
    print(f'{name:<18} MAE=${mae:.4f}  Skill={skill:+.2f}%')

print(f"\nPersistence MAE : ${mae_naive:.4f}  Skill=0.00%")

In [ ]:
# ── Task 4b: Moving-block bootstrap confidence intervals ─────────────────────

def moving_block_bootstrap_skill(model_errs, naive_errs, block_size=10, n_resamples=1000, seed=42):
    """
    95% CI for Skill = 1 - E[model_err] / E[naive_err]
    using a moving-block bootstrap to preserve temporal autocorrelation.
    """
    rng = np.random.default_rng(seed)
    n   = len(model_errs)
    skills = []
    for _ in range(n_resamples):
        # Draw blocks of consecutive dates
        starts = rng.integers(0, n - block_size + 1, size=n // block_size + 1)
        idx    = np.concatenate([np.arange(s, min(s + block_size, n)) for s in starts])[:n]
        m_mae  = model_errs[idx].mean()
        p_mae  = naive_errs[idx].mean()
        skills.append(compute_skill(m_mae, p_mae))
    skills = np.array(skills)
    return np.percentile(skills, [2.5, 97.5]), np.mean(skills)


print('Computing bootstrap intervals (1,000 resamples each) ...')
boot_results = {}
for name, res in results.items():
    ci, boot_mean = moving_block_bootstrap_skill(
        res['model_errs'], res['naive_errs'], block_size=10, n_resamples=1000
    )
    boot_results[name] = {'skill': res['skill'], 'ci_lo': ci[0], 'ci_hi': ci[1]}
    includes_zero = ci[0] <= 0 <= ci[1]
    print(f"{name:<18} Skill={res['skill']:+.2f}%  95% CI [{ci[0]:+.2f}%, {ci[1]:+.2f}%]  "
          f"{'⚠️  includes 0' if includes_zero else '✅ excludes 0'}") 

# Plot
fig, ax = plt.subplots(figsize=(10, 4))
names = list(boot_results.keys())
y_pos = np.arange(len(names))
for i, name in enumerate(names):
    r = boot_results[name]
    ax.plot([r['ci_lo'], r['ci_hi']], [i, i], 'o-', color='steelblue', lw=2)
    ax.scatter(r['skill'], i, color='navy', zorder=5, s=50)
ax.axvline(0, color='red', ls='--', lw=1.5, label='Persistence (Skill=0)')
ax.set_yticks(y_pos)
ax.set_yticklabels(names)
ax.set_xlabel('Skill Score (%)')
ax.set_title('AAPL Validation — 95% Moving-Block Bootstrap Intervals\n(dot = point estimate; line = 95% CI)')
ax.legend()
plt.tight_layout()
plt.show()

print('\n❓ Discussion question:')
print('   The best model has a positive skill score. Its interval includes zero.')
print('   Would you call it a reliable winner? Why or why not?')

In [ ]:
# ✅ TEACHER NOTES — Task 4
#
# Expected student reasoning:
#   "No — the interval includes zero, meaning the improvement is consistent with
#    chance. We cannot rule out that the apparent gain is due to random variation
#    in the validation sample."
#
# Why moving-block (not i.i.d.) bootstrap?
#   Daily returns exhibit autocorrelation. If we resample individual dates
#   independently, we break that structure and underestimate uncertainty.
#   Block size 10 ≈ two calendar weeks, which preserves short-run correlation.
#
# Key references:
#   - Bergmeir, Hyndman & Koo (2018) on time-series cross-validation
#   - Roberts et al. (2017) on temporal structure in CV
#
# Scaling ablation teaching point:
#   Ask students to also run models WITHOUT scaling (P0) and compare.
#   Random Forest and Hist. GBT should produce IDENTICAL predictions (tree
#   splits on rank order, not absolute values).
#   SVR and Ridge will differ (distance- and penalty-based).
print('Teacher notes printed above.')

### 🔎 Bonus: Scaling Ablation
Run the same models **without** standardisation (P0 — raw features) and compute the change in MAE.
Which models are affected? Does the direction match your prediction?

In [ ]:
# ── Scaling ablation (P0 vs P2) ───────────────────────────────────────────────
print('P0 = raw features | P2 = standardised features')
print(f'{"Model":<18} {"MAE P0":>9} {"MAE P2":>9} {"ΔMAE(P2-P0)":>12} {"Tree-invariant?":>15}')
print('-' * 70)

tree_models = {'Random Forest', 'Hist. GBT'}
for name, model_cls in [
    ('Linear',        LinearRegression()),
    ('Ridge',         Ridge(alpha=1.0, random_state=RANDOM_STATE)),
    ('SVR',           SVR(kernel='rbf', C=10.0, epsilon=0.01)),
    ('Random Forest', RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE)),
    ('Hist. GBT',     HistGradientBoostingRegressor(random_state=RANDOM_STATE)),
]:
    # P0: raw features
    m_p0 = model_cls.__class__(**model_cls.get_params())
    m_p0.fit(X_train.values, y_train)
    n       = min(len(X_val), len(anchor_val), len(actual_val))
    pred_p0 = anchor_val[:n] + m_p0.predict(X_val.values[:n])
    mae_p0  = mean_absolute_error(actual_val[:n], pred_p0)

    # P2: standardised features (already computed above)
    m_p2 = model_cls.__class__(**model_cls.get_params())
    m_p2.fit(X_train_p2, y_train)
    pred_p2 = anchor_val[:n] + m_p2.predict(X_val_p2[:n])
    mae_p2  = mean_absolute_error(actual_val[:n], pred_p2)

    delta      = mae_p2 - mae_p0
    invariant  = '✅ Yes' if abs(delta) < 1e-9 else '— No'
    print(f'{name:<18} {mae_p0:>9.4f} {mae_p2:>9.4f} {delta:>+12.6f} {invariant:>15}')

print('\n💡 Tree models (Random Forest, Hist. GBT) should show ΔMAE ≈ 0.')
print('   They split on rank order, which is preserved by any monotone transformation.')

---
## 🎯 Task 5 · Transfer the Framework to a New Domain
*Estimated time: 10 minutes*

The three questions we practised are domain-agnostic:

| # | Question |
|---|----------|
| 1 | What information could the model see at prediction time? |
| 2 | How well does a simple rule do on the same examples? |
| 3 | How certain is the improvement? |

**Scenario:** A school claims its AI attendance predictor achieves R² = 0.91.

Answer the questions below **without writing any code** — think conceptually.

### Your answers

**Q1 — What information was available at prediction time?**

*Write here: What data could a school plausibly have on Monday morning before attendance is taken?*

> *(your answer)*

---

**Q2 — What is the simplest baseline for attendance prediction?**

*Write here: What is the equivalent of "persistence" for attendance? e.g., "predict today's attendance = yesterday's attendance"?*

> *(your answer)*

---

**Q3 — How would you measure the uncertainty of the R² = 0.91 claim?**

*Write here: What information would you need? Would you use the same bootstrap approach?*

> *(your answer)*

---

**Q4 — Design a leakage check for the attendance model.**

*Write here: What features might accidentally "look ahead"? e.g., using the week's total absence rate as a feature for Monday?*

> *(your answer)*

In [ ]:
# ✅ TEACHER REFERENCE ANSWERS — Task 5
#
# Q1 — Available information:
#   Past attendance records, day of week, term calendar (exams, holidays),
#   weather forecasts for the day, known sports fixtures, teacher absences.
#   NOT available: today's actual absence before it is recorded.
#
# Q2 — Simple baseline:
#   "Predict today's attendance = last week's attendance on the same day of week"
#   OR "use the class average over the past 4 weeks".
#   These are the attendance equivalent of persistence.
#
# Q3 — Uncertainty:
#   Would need: test period size, autocorrelation of daily attendance,
#   how many schools / classes were included.
#   The same block-bootstrap applies if data is a time series.
#   If data is a cross-section of students, use ordinary bootstrap.
#
# Q4 — Leakage check:
#   Leaky features might include:
#   - Weekly absence rate that includes today's row
#   - Term-level statistics (computed over all days in the term, including future)
#   - Teacher's subjective annotation written AFTER seeing the day's attendance
#   The "absurd future values" check translates to: append fake attendance for
#   future days, rebuild features, verify past rows are unchanged.
print('Teacher reference answers printed above in comments.')

---
## 📋 Lesson Summary

| Concept | What you practised |
|---------|--------------------|
| **Naive baseline** | Persistence (copy last price) — the floor every model must beat |
| **R² trap** | A high R² does not mean the model learned useful change signals |
| **Data leakage** | Six leakage types; the "absurd future" check |
| **10 legal features** | All relative, all windows ending at today |
| **Target = change** | $\hat{P}_{t+1} = P_t + \widehat{\Delta P}$ removes price-level confusion |
| **Skill Score** | $1 - \text{MAE}_{\text{model}} / \text{MAE}_{\text{persistence}}$ |
| **Bootstrap CI** | Moving-block; all 20 project intervals included zero |
| **Scaling sensitivity** | Tree models invariant; SVR / Ridge sensitive |
| **Domain transfer** | Same 3 questions apply to any AI prediction claim |

### The three questions you can take anywhere

1. ✅ **Information:** What could the model see at prediction time?
2. ✅ **Baseline:** How well does the simplest rule do on the same examples?
3. ✅ **Certainty:** How uncertain is the improvement?

---

### References

1. Pedregosa et al. (2011). Scikit-learn. *JMLR* 12, 2825–2830. https://jmlr.org/papers/v12/pedregosa11a.html
2. Hastie, Tibshirani & Friedman (2009). *The Elements of Statistical Learning* (2nd ed.). Springer.
3. Breiman (2001). Random Forests. *Machine Learning* 45(1), 5–32.
4. Drucker et al. (1997). Support Vector Regression Machines. *NIPS* 9, 155–161.
5. Friedman (2001). Gradient Boosting Machine. *Annals of Statistics* 29(5), 1189–1232.
6. Bergmeir, Hyndman & Koo (2018). Cross-Validation for Time Series. *CSDA* 120, 70–83.
7. Kaufman et al. (2012). Leakage in Data Mining. *ACM TKDD* 6(4), 1–21.
8. Fama (1970). Efficient Capital Markets. *Journal of Finance* 25(2), 383–417.
9. Hyndman & Athanasopoulos (2021). *Forecasting: Principles and Practice* (3rd ed.). https://otexts.com/fpp3/

**Data sources:**
- AAPL: StockAnalysis API — `GET https://stockanalysis.com/api/symbol/s/aapl/history`
- BTC:  Binance REST API  — `GET https://api.binance.com/api/v3/klines?symbol=BTCUSDT&interval=1d`

---
*INT4096 Group Project · Zhang Yancheng, Dong Yuankun, Zhang Haoyu, Lau Pak Hei, Lui Shing Lam*  
*The Education University of Hong Kong, Semester 1, 2026–27*  
*Test block status: 🔒 SEALED (Decision D7, Pending human approval)*
